# מבצע StarLadder: סימולציית Double Elimination חיה

מחברת זו מתעדת את ריצת הקדם־ייצור המבוקרת של StarLadder StarSeries Fall 2026. היא משתמשת בארטיפקט המכויל שנבדק מחדש על Test, בונה תמונת Elo קפואה מכל המידע שהיה זמין לפני פתיחת האירוע, ומריצה שתי סימולציות בלתי־תלויות של 100,000 טורנירים. כל הרצה נשמרת עם provenance מלא כדי שניתן יהיה לשחזר במדויק איזה מודל, seed, מבנה טורניר ומאגר מפות יצרו כל תוצאה. לאחר שהריצה עברה את שער היציבות, Notebook 1.8 הרחיב את אותו צינור למיליון איטרציות לכל seed והוסיף אנליטיקה של matchups, סגניות ופודיומים.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elo_module_dir = PROJECT_ROOT / 'src' / 'features'
if str(elo_module_dir) not in sys.path:
    sys.path.insert(0, str(elo_module_dir))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from elo import PointInTimeEngine
from src.models.simulator import FrozenEloSnapshot
from src.models.bracket_simulator import (
    N_ELO_FEATURES_EXPECTED_ORDER,
    TournamentEloState,
    print_tournament_summary,
    simulate_double_elimination_tournament,
)
from src.models.tournament_exporter import export_tournament_results

## תצורת הטורניר הנעולה

ארבעת משחקי הפתיחה מוזנים בסדר הרשמי, משום שהסדר קובע אילו נתיבים נפגשים בהמשך ה־Upper וה־Lower Bracket. כל המשחקים הם Bo3, למעט Grand Final שהוא Bo5. המנוע תומך גם ב־bracket reset, אך הוא כבוי כאן: לוח StarLadder/HLTV מפרסם Grand Final יחיד ולא סדרת reset נוספת. פיצ'ר זה נשאר זמין עבור אירועים שבהם התקנון דורש מהעולה מה־Lower Bracket לנצח שתי סדרות גמר.

סדרי veto עתידיים אינם ידועים, ולכן בכל matchup נדגם סדר מפות ללא החזרה מתוך מאגר המפות הפעיל. זוהי הנחת אי־ודאות מפורשת ולא תחליף למודל veto.

In [2]:
EVENT_START = pd.Timestamp('2026-09-17 00:00:00')
K_FACTOR = 24.0
N_ITERATIONS = 100_000
BATCH_SIZE = 2_048
SEEDS = (42, 99)
GRAND_FINAL_RESET = False
OPENING_MATCHUPS = (
    ('mouz', 'nrg'),
    ('vitality', 'magic'),
    ('natus vincere', 'aurora'),
    ('furia', 'mibr'),
)
ACTIVE_MAP_POOL = (
    'Cache', 'Dust2', 'Mirage', 'Inferno', 'Nuke', 'Ancient', 'Anubis'
)
HISTORICAL_MAP_ORDERS = {}
MODEL_ARTIFACT_PATH = (
    PROJECT_ROOT / 'artifacts' / 'map_classifier'
    / 'canonical_elo_isotonic.joblib'
)
TOURNAMENT_TEAMS = tuple(
    team for matchup in OPENING_MATCHUPS for team in matchup
)
assert len(TOURNAMENT_TEAMS) == len(set(TOURNAMENT_TEAMS)) == 8

## בניית תמונת Elo נקודתית בזמן ואימות הארטיפקט

המנוע עובר כרונולוגית רק על מפות שהסתיימו לפני פתיחת הטורניר. דירוגי ה־Global וה־Map שמתקבלים לאחר המעבר מוקפאים ומועתקים מחדש לכל מסלול Monte Carlo. בתוך מסלול יחיד העדכונים חיים ועוברים עם הקבוצה בין ה־Upper וה־Lower Bracket; בין מסלולים אין זליגת מצב.

לפני הסימולציה נבדק שכל שמונה המפתחות הקנוניים קיימים בהיסטוריה ושאף קבוצה אינה נופלת בשקט ל־1500. בנוסף, בנאי הנתיב המהיר מאמת שסדר ששת הפיצ'רים ב־Booster זהה בדיוק ל־allowlist הנעול לפני שהוא מקבל מטריצות NumPy חסרות שמות עמודות.

In [3]:
raw_df = pd.read_csv(
    PROJECT_ROOT / 'data' / 'final_tournament_features.csv',
    low_memory=False,
)
raw_datetimes = pd.to_datetime(raw_df['datetime'], errors='raise')
history = raw_df.loc[raw_datetimes < EVENT_START].copy()
if history.empty:
    raise AssertionError('לא נמצאה היסטוריה לפני מועד הטורניר.')
engine = PointInTimeEngine(k_factor=K_FACTOR)
engine.transform(history)
snapshot = FrozenEloSnapshot(
    global_ratings=dict(engine.global_ratings_),
    map_ratings=dict(engine.map_ratings_),
    initial_rating=engine.initial_rating,
)
starting_state = TournamentEloState.from_snapshot(
    snapshot, TOURNAMENT_TEAMS, ACTIVE_MAP_POOL, k_factor=K_FACTOR
)
starting_ratings = pd.DataFrame({
    'קבוצה': TOURNAMENT_TEAMS,
    'Elo גלובלי קפוא': [
        starting_state.global_rating(team) for team in TOURNAMENT_TEAMS
    ],
}).sort_values('Elo גלובלי קפוא', ascending=False).reset_index(drop=True)
if starting_ratings['Elo גלובלי קפוא'].eq(snapshot.initial_rating).any():
    raise AssertionError('קבוצה נפלה לדירוג ברירת המחדל 1500.')

model_bundle = joblib.load(MODEL_ARTIFACT_PATH)
assert model_bundle['feature_columns'] == list(N_ELO_FEATURES_EXPECTED_ORDER)
deployed_model = model_bundle['calibrated_model']
print(f'מפת היסטוריה אחרונה: {raw_datetimes.loc[history.index].max()}')
print(f'מספר קבוצות היסטוריות: {len(snapshot.global_ratings):,}')
starting_ratings

מפת היסטוריה אחרונה: 2026-06-21 18:00:00
מספר קבוצות היסטוריות: 270


,קבוצה,Elo גלובלי קפוא
0,vitality,1868.139017
1,natus vincere,1776.148668
2,furia,1739.099059
3,aurora,1718.411910
4,mouz,1686.323519
5,magic,1577.033504
6,mibr,1528.695380
7,nrg,1351.651965


## שתי ריצות Monte Carlo בקנה מידה מלא

בכל seed מורצים 100,000 טורנירים. פס ההתקדמות מציג קצב איטרציות לשנייה ו־ETA בפורמט קבוע. העמודה `P(SF)` מייצגת את ההסתברות להישאר בין ארבע הקבוצות האחרונות לאחר Lower Bracket Semi-finals; `P(Final)` מייצגת הגעה ל־Grand Final. כך סכומי המשתתפים הם בדיוק 8, 4, 2 ו־1 בהתאמה, וההסתברויות מונוטוניות לכל קבוצה.

In [4]:
run_counts = {}
run_seconds = {}
run_summaries = {}
for seed in SEEDS:
    started = perf_counter()
    run_counts[seed] = simulate_double_elimination_tournament(
        OPENING_MATCHUPS,
        starting_state,
        deployed_model,
        ACTIVE_MAP_POOL,
        historical_map_orders=HISTORICAL_MAP_ORDERS,
        n_iterations=N_ITERATIONS,
        random_state=seed,
        batch_size=BATCH_SIZE,
        grand_final_reset=GRAND_FINAL_RESET,
        show_progress=True,
    )
    run_seconds[seed] = perf_counter() - started
    print(f'זמן ריצה seed={seed}: {run_seconds[seed]:.3f} שניות')
    run_summaries[seed] = print_tournament_summary(
        run_counts[seed], N_ITERATIONS
    )

Double-Elimination Monte Carlo:   0%|          | 0/100000 [00:00<?, ?iter/s]

זמן ריצה seed=42: 122.106 שניות
         Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
     Vitality 100.0% 87.7%    67.7%       44.3%         0.2%
Natus Vincere 100.0% 73.2%    42.3%       20.5%         0.1%
        FURIA 100.0% 70.5%    34.5%       14.8%         0.1%
       Aurora 100.0% 56.0%    23.0%        9.4%         0.1%
         MOUZ 100.0% 58.9%    23.8%        9.1%         0.1%
        magic 100.0% 30.3%     5.3%        1.3%         0.0%
         MIBR 100.0% 17.7%     3.0%        0.6%         0.0%
          NRG 100.0%  5.8%     0.4%        0.0%         0.0%


Double-Elimination Monte Carlo:   0%|          | 0/100000 [00:00<?, ?iter/s]

זמן ריצה seed=99: 138.758 שניות
         Team  P(QF) P(SF) P(Final) P(Champion) SE(Champion)
     Vitality 100.0% 87.7%    67.9%       44.6%         0.2%
Natus Vincere 100.0% 73.0%    42.2%       20.2%         0.1%
        FURIA 100.0% 70.3%    34.5%       15.0%         0.1%
       Aurora 100.0% 55.6%    22.7%        9.3%         0.1%
         MOUZ 100.0% 59.1%    23.7%        9.0%         0.1%
        magic 100.0% 30.4%     5.5%        1.3%         0.0%
         MIBR 100.0% 18.0%     3.1%        0.7%         0.0%
          NRG 100.0%  5.9%     0.3%        0.0%         0.0%


## שער יציבות אמפירי

אנו משווים לכל קבוצה את `P(Champion)` בין seed 42 ל־seed 99. ההרצה מאושרת רק אם ההפרש המוחלט המרבי קטן או שווה לחצי נקודת אחוז. הבדיקה מודדת רעש Monte Carlo; היא אינה רווח סמך לאי־הוודאות של המודל עצמו.

In [5]:
champion_probabilities = {
    seed: {
        team: counts['Champion'] / N_ITERATIONS
        for team, counts in run_counts[seed].items()
    }
    for seed in SEEDS
}
stability = pd.DataFrame({
    'Team': TOURNAMENT_TEAMS,
    'P(Champion), seed=42': [champion_probabilities[42][t] for t in TOURNAMENT_TEAMS],
    'P(Champion), seed=99': [champion_probabilities[99][t] for t in TOURNAMENT_TEAMS],
})
stability['הפרש מוחלט'] = (
    stability['P(Champion), seed=42']
    - stability['P(Champion), seed=99']
).abs()
max_champion_difference = stability['הפרש מוחלט'].max()
assert max_champion_difference <= 0.005, (
    f'שער היציבות נכשל: {max_champion_difference:.4%}'
)
print(f'הפרש P(Champion) מרבי: {max_champion_difference:.4%}')
stability.sort_values('הפרש מוחלט', ascending=False)

הפרש P(Champion) מרבי: 0.3300%


,Team,"P(Champion), seed=42","P(Champion), seed=99",הפרש מוחלט
4,natus vincere,0.20513,0.20183,0.00330
2,vitality,0.44286,0.44594,0.00308
5,aurora,0.09411,0.09272,0.00139
6,furia,0.14830,0.14968,0.00138
0,mouz,0.09057,0.08970,0.00087
3,magic,0.01251,0.01321,0.00070
7,mibr,0.00610,0.00652,0.00042
1,nrg,0.00042,0.00040,0.00002


## יצוא אטומי ושרשרת provenance

כל seed נשמר כ־CSV, JSON וקובץ metadata תחת `results/starladder/`. כל קובץ נכתב תחילה ל־`.tmp` ומועבר לשם הסופי רק לאחר כתיבה מוצלחת; קובץ metadata נוחת אחרון ומשמש סימן לכך שהשלישייה הושלמה. טביעת SHA־256 קושרת את התחזית לביטים המדויקים של ארטיפקט המודל, ו־`runs_index.csv` מרכז את היסטוריית הריצות מבלי למחוק תוצרים קודמים.

In [6]:
exports = {}
for seed in SEEDS:
    exports[seed] = export_tournament_results(
        run_summaries[seed],
        'starladder',
        PROJECT_ROOT / 'results',
        seed,
        N_ITERATIONS,
        MODEL_ARTIFACT_PATH,
        k_factor=K_FACTOR,
        active_map_pool=ACTIVE_MAP_POOL,
        quarterfinals=OPENING_MATCHUPS,
        historical_map_orders=HISTORICAL_MAP_ORDERS,
        execution_seconds=run_seconds[seed],
        extra_metadata={
            'format': 'eight_team_double_elimination',
            'grand_final_best_of': 5,
            'other_matches_best_of': 3,
            'grand_final_reset': GRAND_FINAL_RESET,
            'elo_snapshot_cutoff': EVENT_START.isoformat(),
            'stage_semantics': {
                'QF': 'entered tournament',
                'SF': 'survived to final four after lower semi-finals',
                'Final': 'reached Grand Final',
            },
            'paired_stability_seed': 99 if seed == 42 else 42,
            'max_champion_probability_difference': float(max_champion_difference),
        },
    )
    print(f'seed={seed}: {exports[seed].metadata_path.relative_to(PROJECT_ROOT)}')

seed=42: results\starladder\starladder_20260915T120101Z_seed42_n100000_metadata.json
seed=99: results\starladder\starladder_20260915T120101Z_seed99_n100000_metadata.json


## פרשנות תפעולית

הטבלאות הן תחזית טרום־טורניר המבוססת על Elo היסטורי, כיול איזוטוני ודגימת מפות אחידה. הן אינן יודעות את בחירות ה־veto, שינויים ברוסטר או מידע שהתרחש לאחר חיתוך הדאטה. לכן `SE(Champion)` מתאר רק את דיוק האמידה של Monte Carlo, ואינו מכסה אי־ודאות במודל או במידע החסר. תוצרים אלה הם baseline תפעולי שניתן לעדכן בהמשך עם תוצאות אמת ומפות ידועות, תוך שמירת כל ריצה קודמת באינדקס.